# Average Depth Estimation — Cross Grade Slope

Upload 4 CSV files from monocular depth estimation runs and average them by image filename.

## Step 1: Upload the 4 CSVs

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
csv_names = sorted(uploaded.keys())
print(f"Uploaded {len(csv_names)} files: {csv_names}")

## Step 2: Load and preview each CSV

In [ ]:
dfs = {}
for name in csv_names:
    df = pd.read_csv(name)
    dfs[name] = df
    print(f"\n--- {name} ---")
    print(f"  {len(df)} rows, columns: {list(df.columns)}")
    display(df.head())

## Step 3: Combine and average by filename

Numeric columns are averaged across all runs for each image. The `count` column shows how many of the 4 CSVs contained that filename.

In [ ]:
import re

numeric_cols = ['trail_coverage_pct', 'cross_grade_slope_deg', 'cross_grade_slope_pct', 'roll_correction_deg']

def normalize_filename(name):
    """Strip download-duplicate suffixes like ' (1)', ' (2)' etc. from filenames."""
    return re.sub(r'\s*\(\d+\)', '', name)

# Tag each dataframe with its source, normalize filenames, and concatenate
all_dfs = []
for name, df in dfs.items():
    tagged = df.copy()
    tagged['source'] = name
    tagged['filename'] = tagged['filename'].apply(normalize_filename)
    all_dfs.append(tagged)

combined = pd.concat(all_dfs, ignore_index=True)

# Average numeric columns by normalized filename, count how many runs included each image
averaged = combined.groupby('filename').agg(
    trail_coverage_pct=('trail_coverage_pct', 'mean'),
    cross_grade_slope_deg=('cross_grade_slope_deg', 'mean'),
    cross_grade_slope_pct=('cross_grade_slope_pct', 'mean'),
    roll_correction_deg=('roll_correction_deg', 'mean'),
    count=('source', 'nunique')
).reset_index()

# Sort by filename
averaged = averaged.sort_values('filename').reset_index(drop=True)

print(f"{len(averaged)} unique images averaged across runs")
averaged

## Step 4: Per-run comparison

Standard deviation across runs for each image — shows how consistent the estimates are.

In [ ]:
std_df = combined.groupby('filename')[numeric_cols].std().reset_index()
std_df.columns = ['filename'] + [c + '_std' for c in numeric_cols]
std_df = std_df.sort_values('filename').reset_index(drop=True)

print("Standard deviation across runs per image:")
std_df

## Step 5: Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Cross grade slope (deg) — each run as a thin line, average as bold
for name, df in dfs.items():
    df_norm = df.copy()
    df_norm['filename'] = df_norm['filename'].apply(normalize_filename)
    df_sorted = df_norm.sort_values('filename')
    axes[0].plot(range(len(df_sorted)), df_sorted['cross_grade_slope_deg'].values,
                 alpha=0.35, linewidth=1, label=name)
axes[0].plot(range(len(averaged)), averaged['cross_grade_slope_deg'].values,
             color='black', linewidth=2, label='Average')
axes[0].set_ylabel('Cross Grade Slope (deg)')
axes[0].set_title('Cross Grade Slope per Image — Individual Runs vs Average')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Trail coverage — same treatment
for name, df in dfs.items():
    df_norm = df.copy()
    df_norm['filename'] = df_norm['filename'].apply(normalize_filename)
    df_sorted = df_norm.sort_values('filename')
    axes[1].plot(range(len(df_sorted)), df_sorted['trail_coverage_pct'].values,
                 alpha=0.35, linewidth=1, label=name)
axes[1].plot(range(len(averaged)), averaged['trail_coverage_pct'].values,
             color='black', linewidth=2, label='Average')
axes[1].set_ylabel('Trail Coverage (%)')
axes[1].set_title('Trail Coverage per Image — Individual Runs vs Average')
axes[1].set_xlabel('Image index (sorted by filename)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Export averaged results

In [ ]:
output = averaged[['filename', 'trail_coverage_pct', 'cross_grade_slope_deg',
                    'cross_grade_slope_pct', 'roll_correction_deg', 'count']].copy()

output.to_csv('averaged_depth_estimation.csv', index=False)
files.download('averaged_depth_estimation.csv')
print(f"Exported {len(output)} rows to averaged_depth_estimation.csv")